# **((Data Cleaning))**

## Objectives

* "We will check all the images and make sure we dont have any data represented as text or any corrupted files"
* "We will also combine all the images into one file and then split them all into our train, validation, and test sets."

## Inputs

* inputs/skin_cancer_dataset/melanoma_cancer_dataset/train
* inputs/skin_cancer_dataset/melanoma_cancer_dataset/validation
* inputs/skin_cancer_dataset/melanoma_cancer_dataset/test

## Outputs

No files will be placed in outputs, however, we will be forming the train, validation, and test folders under inputs/skin_cancer_dataset/melanoma_cancer_dataset

## Additional Comments

No additional comments

---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [1]:
import os
current_dir = os.getcwd()
current_dir

'/workspaces/skin-lesion-detector/jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [2]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [3]:
current_dir = os.getcwd()
current_dir

'/workspaces/skin-lesion-detector'

---

# Data Cleaning

## Excluding Files based on their extension

Here, we will check all the images and make sure we don't have any data represented as text

In [4]:
from pathlib import Path
def clean_skin_cancer_dataset(my_data_dir):
    allowed_extensions = {'.png', '.jpeg', '.jpg'}
    class_names = ['benign', 'malignant']
    results = {}

    for class_name in class_names:
        class_dir = Path(my_data_dir) / class_name

        if not class_dir.exists():
            results[class_names] = {'status': 'folder not found', 'kept': 0, 'removed': 0}
            continue

        kept = 0
        removed = 0

        for file_path in class_dir.rglob('*'):
            if file_path.is_file():
                if file_path.suffix.lower() in allowed_extensions:
                    kept += 1
                else:
                    file_path.unlink()
                    removed += 1

        results[class_name] = {'status': 'cleaned', 'kept': kept, 'removed': removed}

    return results

train_result = clean_skin_cancer_dataset('inputs/skin_cancer_dataset/melanoma_cancer_dataset/train')
test_result  = clean_skin_cancer_dataset('inputs/skin_cancer_dataset/melanoma_cancer_dataset/test')

all_results = {'train': train_result, 'test': test_result}
print(all_results)


{'train': {'benign': {'status': 'cleaned', 'kept': 5000, 'removed': 0}, 'malignant': {'status': 'cleaned', 'kept': 4605, 'removed': 0}}, 'test': {'benign': {'status': 'cleaned', 'kept': 500, 'removed': 0}, 'malignant': {'status': 'cleaned', 'kept': 500, 'removed': 0}}}


As we can see, there are no files without the extensions .jpeg, .jpg, .png

## Removing corrupted files

Here, we will be removing images that are corrupted or cannot be opened

In [5]:
from pathlib import Path
from PIL import Image, UnidentifiedImageError


def remove_corrupt_images(data_dir):
    class_names = ['benign', 'malignant']
    results = {}

    for class_name in class_names:
        class_dir = Path(data_dir) / class_name
        removed_files = []

        if not class_dir.exists():
            results[class_name] = {
                'status': 'folder not found',
                'removed': removed_files,
            }
            continue

        for file_path in class_dir.iterdir():
            if not file_path.is_file():
                continue

            try:
                with Image.open(file_path) as image:
                    image.verify()
            except (UnidentifiedImageError, OSError, SyntaxError):
                file_path.unlink()
                removed_files.append(str(file_path))

        results[class_name] = {
            'status': 'cleaned',
            'removed': removed_files,
        }

    return results


corrupt_train_set_result = clean_skin_cancer_dataset('inputs/skin_cancer_dataset/melanoma_cancer_dataset/train')
corrupt_test_set_result  = clean_skin_cancer_dataset('inputs/skin_cancer_dataset/melanoma_cancer_dataset/test')

all_corrupt_results = {'train': corrupt_train_set_result, 'test': corrupt_test_set_result}
print(all_corrupt_results)


{'train': {'benign': {'status': 'cleaned', 'kept': 5000, 'removed': 0}, 'malignant': {'status': 'cleaned', 'kept': 4605, 'removed': 0}}, 'test': {'benign': {'status': 'cleaned', 'kept': 500, 'removed': 0}, 'malignant': {'status': 'cleaned', 'kept': 500, 'removed': 0}}}


As we can see, there were no corrupt files either

# Splitting dataset into train, validation, and test sets

Since we halready have the test and train sets, I would like to move them all into one folder so I can make sure all images are split correctly into their respective sets

In [6]:
os.makedirs('inputs/skin_cancer_dataset/melanoma_cancer_dataset/validation', exist_ok=True)
os.makedirs('inputs/skin_cancer_dataset/melanoma_cancer_dataset/placeholder', exist_ok=True)

Now, we will combine the folders with the same label(or class) and put them in the placeholder folder for proper distribution

In [ ]:
from pathlib import Path
import shutil

base_dir = Path('inputs/skin_cancer_dataset/melanoma_cancer_dataset')
source_folders = ('train', 'test')
classes = ('benign', 'malignant')
placeholder_dir = base_dir / 'placeholder'

for class_name in classes:
    destination_dir = placeholder_dir / class_name
    destination_dir.mkdir(parents=True, exist_ok=True)

    for source_folder in source_folders:
        source_dir = base_dir / source_folder / class_name
        if not source_dir.exists():
            continue

        for source_file in source_dir.iterdir():
            if not source_file.is_file():
                continue

            destination_file = destination_dir / source_file.name
            if destination_file.exists():
                stem = source_file.stem
                suffix = source_file.suffix
                counter = 1
                while destination_file.exists():
                    destination_file = destination_dir / f'{stem}_{counter}{suffix}'
                    counter += 1

            shutil.move(str(source_file), str(destination_file))

print(f'Combined train and test images in {placeholder_dir}')

Combined train and test images in inputs/skin_cancer_dataset/melanoma_cancer_dataset/placeholder


Now, we will separate all the images into three parts. Training, validation, and test sets